<a href="https://colab.research.google.com/github/kiran9846/oof-fraudguard-anomaly-detection/blob/main/copy_of_Ctrl_alt_elite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Inspect Data



In [ ]:
import pandas as pd
df = pd.read_csv('/content/sample_data/data.csv')
df.head()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Summary Statistics

In [ ]:
# Quantitative
quantitative = df[["TransactionAmount", "CustomerAge", "TransactionDuration",
    "LoginAttempts", "AccountBalance"]]

quantitative.describe()

In [ ]:
# Categorical
categorical = df[df.columns.difference(quantitative.columns)]
for col in categorical:
    print(f"=== {col} ===")
    print(df[col].value_counts())
    print()

**Pipeline: **
1. Correlation Matrix
         ↓
2. Feature Engineering
   (create new meaningful features)
         ↓
3. Scale ALL features
         ↓
4. Anomaly Detection
   (Isolation Forest or K-Means)
         ↓
5. Correlation Analysis
         ↓
6. Regression Analysis
         ↓
7. Predictive Modeling
         ↓
8. Evaluation

Priorities:
Correlation
Predictive Modeling

Correlation matrix(Done)

In [ ]:
from sklearn.cluster import KMeans
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

In [ ]:
numerical_cols = df.select_dtypes(include='number').columns.tolist()
print(numerical_cols)

In [ ]:
corr = df[numerical_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f")
plt.title('Correlation Matrix')
plt.show()

Feature Engineering

In [ ]:
df['TransactionAmount'] = np.log1p(df['TransactionAmount'])
df['Account_Transaction_Count'] = df.groupby('AccountID')['AccountID'].transform('count')

df['Account_Avg_Amount'] = df.groupby('AccountID')['TransactionAmount'].transform('mean')
df['Amount_to_Balance'] = df['TransactionAmount'] / (df['AccountBalance'] + 1)
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'])
df['PreviousTransactionDate'] = pd.to_datetime(df['PreviousTransactionDate'])
df['Hour'] = df['TransactionDate'].dt.hour
df['Weekday'] = df['TransactionDate'].dt.weekday
df['IsWeekend'] = df['Weekday'].isin([5,6]).astype(int)
df['Accounts_per_Device'] = df.groupby('DeviceID')['AccountID'].transform('nunique')
df.head(5)

In [ ]:
plt.figure(figsize=(6, 4))

palette = sns.color_palette("tab20")

ax = sns.countplot(data=df, x='Accounts_per_Device', palette=palette)

plt.title('Accounts_per_Device Distribution')
plt.xlabel('Number_Accounts_per_Device')
plt.ylabel('Number_of_Devices')

plt.axvline(x=4.5, color='red', linestyle='--', label='Risk Threshold')
plt.legend()
plt.tight_layout()
plt.show()

From this graph we can find out the number of accounts per devices.

**Select Features and Scaling**

In [ ]:
features = df.select_dtypes(include='number').columns.tolist()
print(features)

In [ ]:
X = df[features]
scaler = StandardScaler()
X = X.fillna(X.mean())
X_scaled = scaler.fit_transform(X)

**Isolation Forest on scaled data**

In [ ]:
from sklearn.ensemble import IsolationForest

In [ ]:
# iso = IsolationForest(contamination=0.05, random_state=42)
# iso.fit(X_scaled)

# preds = iso.predict(X_scaled)
# scores = iso.decision_function(X_scaled)

# print("Pred counts:", np.unique(preds, return_counts=True))
# print("Score min/max:", scores.min(), scores.max())

0 means normal and 1 means potential fraud account


In [ ]:
iso = IsolationForest(contamination=0.05, random_state=42)
df['IsFraud'] = (iso.fit_predict(X_scaled) == -1).astype(int)
df['IsFraud'].value_counts()

In [ ]:
counts = df['IsFraud'].value_counts()
labels = ['Normal (0)', 'Suspicious (1)']
colors = ['#2ecc71', '#e74c3c']

plt.figure(figsize=(7, 5))
bars = plt.bar(labels, counts.values, color=colors, width=0.4)

# Value on top of each bar
for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 10,
             str(bar.get_height()),
             ha='center', fontsize=12, fontweight='bold')

plt.title('Normal vs Suspicious Transactions\n(Isolation Forest Results)',
          fontsize=13, fontweight='bold')
plt.xlabel('Transaction Type', fontsize=12)
plt.ylabel('Number of Transactions', fontsize=12)
plt.tight_layout()
plt.show()

**Correlation Analysis**

In [ ]:
correlation_cols = [
    'TransactionAmount', 'CustomerAge', 'TransactionDuration', 'LoginAttempts',
    'AccountBalance', 'Account_Transaction_Count', 'Account_Avg_Amount',
    'Amount_to_Balance', 'Hour', 'Weekday', 'IsWeekend', 'Accounts_per_Device',
    'IsFraud'
]

fraud_analysis_corr = df[correlation_cols].corr()

fraud_drivers = fraud_analysis_corr['IsFraud'].sort_values(ascending=False)

print("Numerical Correlation with Fraud Labels:")
print(fraud_drivers)

plt.figure(figsize=(6, 8))
sns.heatmap(fraud_analysis_corr[['IsFraud']].sort_values(by='IsFraud', ascending=False),
            annot=True, cmap='Reds', fmt=".2f")
plt.title('Feature Correlation with Detected Fraud')
plt.show()

Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

X = df[features]
X = X.fillna(X.mean())
y = df['IsFraud']



X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

y_probs_cv = cross_val_predict(pipeline, X_train, y_train, cv=cv, method='predict_proba')[:, 1]

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
precisions = []
recalls    = []
f1s        = []

print("=== Threshold Comparison (CV) ===\n")
for threshold in thresholds:
    y_pred = (y_probs_cv >= threshold).astype(int)
    report = classification_report(y_train, y_pred,
                                   target_names=['Normal', 'Fraud'],
                                   output_dict=True)
    precisions.append(report['Fraud']['precision'])
    recalls.append(report['Fraud']['recall'])
    f1s.append(report['Fraud']['f1-score'])

    print(f"Threshold {threshold} → "
          f"Precision: {report['Fraud']['precision']:.2f} | "
          f"Recall: {report['Fraud']['recall']:.2f} | "
          f"F1: {report['Fraud']['f1-score']:.2f}")

best_idx       = int(np.argmax(f1s))
best_threshold = thresholds[best_idx]

In [ ]:

plt.figure(figsize=(9, 5))

plt.plot(thresholds, precisions, 'o-', color='#3b82f6', lw=2, ms=8, label='Precision')
plt.plot(thresholds, recalls,    's-', color='#f97316', lw=2, ms=8, label='Recall')
plt.plot(thresholds, f1s,        'D--',color='#8b5cf6', lw=2, ms=8, label='F1 Score')

# Value labels on every point
for i, t in enumerate(thresholds):
    plt.annotate(f'{precisions[i]:.2f}', (t, precisions[i]),
                 textcoords='offset points', xytext=(0, 10),
                 ha='center', fontsize=9, color='#3b82f6')
    plt.annotate(f'{recalls[i]:.2f}',    (t, recalls[i]),
                 textcoords='offset points', xytext=(0, -16),
                 ha='center', fontsize=9, color='#f97316')
    plt.annotate(f'{f1s[i]:.2f}',        (t, f1s[i]),
                 textcoords='offset points', xytext=(6, 4),
                 ha='left',   fontsize=9, color='#8b5cf6')

# Best threshold line
plt.axvline(x=best_threshold, color='#22c55e',
            linestyle='--', lw=1.8, label=f'Best threshold = {best_threshold}')

plt.title('Precision, Recall & F1 at Each Threshold\n(Fraud class — Cross Validation)',
          fontsize=13, fontweight='bold')
plt.xlabel('Threshold', fontsize=11)
plt.ylabel('Score',     fontsize=11)
plt.xticks(thresholds)
plt.ylim(0.3, 1.05)
plt.legend(fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('threshold_chart.png', dpi=150, bbox_inches='tight')
plt.show()

we would rather choose false alarm then missing fraud thats why we use the threshold 0.3

In [ ]:
#Classification report with final threshold
best_threshold = 0.3

pipeline.fit(X_train, y_train)

y_prob_test = pipeline.predict_proba(X_test)[:, 1]
y_pred_test = (y_prob_test >= best_threshold).astype(int)

report_dict = classification_report(y_test, y_pred_test,
                                   target_names=['Normal', 'Fraud'],
                                   output_dict=True)

print(f"=== Final Test Report (Threshold: {best_threshold}) ===")
print(classification_report(y_test, y_pred_test,
      target_names=['Normal', 'Fraud']))



In [ ]:
fraud_stats = report_dict['Fraud']
metrics = ['Precision', 'Recall', 'F1-Score']
scores = [fraud_stats['precision'], fraud_stats['recall'], fraud_stats['f1-score']]

plt.figure(figsize=(9, 5))
sns.set_style("white")

colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
bars = plt.bar(metrics, scores, color=colors, alpha=0.8, edgecolor='black')


for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.02,
             f'{yval:.2f}', ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.ylim(0, 1.1)
plt.title(f'Fraud Detection Performance (Threshold: {best_threshold})', fontsize=14, pad=20)
plt.ylabel('Score (0.0 - 1.0)', fontsize=12)
sns.despine()
plt.show()

In [ ]:
df_test = X_test.copy()
df_test['ActualFraud']      = y_test.values
df_test['FraudProbability'] = y_prob_test
df_test['PredictedFraud']   = y_pred_test

df_test.head()

In [ ]:
plt.figure(figsize=(9, 5))

plt.hist(df_test[df_test['ActualFraud']==0]['FraudProbability'],
         bins=30, alpha=0.6, color='#22c55e', label='Actual Normal')
plt.hist(df_test[df_test['ActualFraud']==1]['FraudProbability'],
         bins=30, alpha=0.6, color='#ef4444', label='Actual Fraud')

plt.axvline(x=0.3, color='black', linestyle='--',
            linewidth=1.8, label='Threshold = 0.3')

plt.title('Fraud Probability Distribution\nDoes the model separate fraud from normal?',
          fontsize=13, fontweight='bold')
plt.xlabel('Fraud Probability Score (0 = safe, 1 = fraud)', fontsize=11)
plt.ylabel('Number of Transactions', fontsize=11)
plt.legend(fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()